# 02d — Evaluation & Hyperparameter Tuning

**Most beginners skip this. Don't.**  
Building a model is easy. Knowing whether it's *actually good* is the hard part.

```
Build model → Evaluate properly → Tune hyperparameters → Repeat
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, make_classification, load_digits
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold,
    GridSearchCV, RandomizedSearchCV, learning_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report, precision_score,
    recall_score, f1_score, roc_curve, auc, precision_recall_curve
)

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100
np.random.seed(42)

---
## 1 — Confusion Matrix

The foundation of classification evaluation. For binary classification:

```
                  Predicted
               Pos      Neg
Actual Pos  [  TP  |  FN  ]     ← FN = missed positives
Actual Neg  [  FP  |  TN  ]     ← FP = false alarms
```

**Medical test analogy**:  
- TP: Has cancer, test says cancer ✓
- FP: Healthy, test says cancer ✗ (false alarm)
- FN: Has cancer, test says healthy ✗ (DANGEROUS)
- TN: Healthy, test says healthy ✓

In [ ]:
cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target

X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

lr = LogisticRegression(max_iter=5000)
lr.fit(X_tr_s, y_tr)
y_pred = lr.predict(X_te_s)

cm = confusion_matrix(y_te, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Malignant', 'Benign'], yticklabels=['Malignant', 'Benign'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix — Raw Counts')

cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_pct, annot=True, fmt='.1%', cmap='Blues', ax=axes[1],
            xticklabels=['Malignant', 'Benign'], yticklabels=['Malignant', 'Benign'])
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title('Confusion Matrix — Percentages')

plt.tight_layout()
plt.show()

---
## 2 — Precision, Recall, F1

Accuracy alone is misleading. These metrics tell you *what kind* of errors you're making.

$$\text{Precision} = \frac{TP}{TP + FP} \quad \text{(of predicted positives, how many are correct?)}$$

$$\text{Recall} = \frac{TP}{TP + FN} \quad \text{(of actual positives, how many did we catch?)}$$

$$\text{F1} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} \quad \text{(harmonic mean)}$$

**When each matters**:
- **Spam filter**: precision matters — don't send real email to spam
- **Cancer detection**: recall matters — don't miss actual cancer
- **Balanced**: F1 balances both

In [ ]:
print(classification_report(y_te, y_pred, target_names=['Malignant', 'Benign']))

print(f"Precision: {precision_score(y_te, y_pred):.4f}")
print(f"Recall:    {recall_score(y_te, y_pred):.4f}")
print(f"F1:        {f1_score(y_te, y_pred):.4f}")

---
## 3 — ROC Curve & AUC

The **ROC curve** plots True Positive Rate vs False Positive Rate at every threshold.

- **AUC = 1.0**: perfect classifier
- **AUC = 0.5**: random guessing (diagonal line)
- **AUC < 0.5**: worse than random

Compare multiple models by overlaying their ROC curves.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=5000),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN (k=5)':           KNeighborsClassifier(n_neighbors=5),
}

plt.figure(figsize=(8, 6))

for name, model in models.items():
    model.fit(X_tr_s, y_tr)
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_te_s)[:, 1]
    else:
        y_prob = model.decision_function(X_te_s)
    fpr, tpr, _ = roc_curve(y_te, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random (AUC = 0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Higher is better')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 4 — Cross-Validation

A single train/test split is **risky** — you might get lucky or unlucky.

**K-Fold Cross-Validation**: split data into k folds, train on k-1, test on 1. Rotate.

```
Fold 1: [TEST] [Train] [Train] [Train] [Train]
Fold 2: [Train] [TEST] [Train] [Train] [Train]
Fold 3: [Train] [Train] [TEST] [Train] [Train]
Fold 4: [Train] [Train] [Train] [TEST] [Train]
Fold 5: [Train] [Train] [Train] [Train] [TEST]
```

**Stratified K-Fold**: maintains class proportions in each fold (important for imbalanced data).

In [ ]:
from sklearn.pipeline import make_pipeline

pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))

cv_scores = cross_val_score(pipe, X_c, y_c, cv=10, scoring='accuracy')

print(f'10-Fold CV scores: {cv_scores.round(4)}')
print(f'Mean:  {cv_scores.mean():.4f}')
print(f'Std:   {cv_scores.std():.4f}')

plt.figure(figsize=(8, 4))
plt.bar(range(1, 11), cv_scores, color='steelblue', edgecolor='black')
plt.axhline(y=cv_scores.mean(), color='r', linestyle='--', label=f'Mean = {cv_scores.mean():.3f}')
plt.fill_between(range(0, 12), cv_scores.mean() - cv_scores.std(), 
                 cv_scores.mean() + cv_scores.std(), alpha=0.2, color='red')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.title('Cross-Validation — Variance across folds')
plt.legend()
plt.xlim(0, 11)
plt.show()

In [ ]:
cv_models = {
    'Logistic': make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    'KNN': make_pipeline(StandardScaler(), KNeighborsClassifier()),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}

cv_results = {}
for name, model in cv_models.items():
    scores = cross_val_score(model, X_c, y_c, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f'{name:20s} → {scores.mean():.4f} ± {scores.std():.4f}')

plt.figure(figsize=(8, 5))
plt.boxplot(cv_results.values(), labels=cv_results.keys())
plt.ylabel('CV Accuracy')
plt.title('Model Comparison with Cross-Validation')
plt.grid(True, alpha=0.3)
plt.show()

---
## 5 — Hyperparameter Tuning

Model parameters are learned during training. **Hyperparameters** are set *before* training.

| Algorithm | Key Hyperparameters |
|-----------|--------------------|
| KNN | k (n_neighbors) |
| Decision Tree | max_depth, min_samples_split |
| Random Forest | n_estimators, max_depth, max_features |
| SVM | C, gamma, kernel |

Two approaches:
- **GridSearchCV**: try all combinations (exhaustive)
- **RandomizedSearchCV**: sample random combinations (faster)

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2'],
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=0)
grid_search.fit(X_tr_s, y_tr)

print(f'Best parameters:  {grid_search.best_params_}')
print(f'Best CV accuracy: {grid_search.best_score_:.4f}')
print(f'Test accuracy:    {grid_search.score(X_te_s, y_te):.4f}')

In [ ]:
from scipy.stats import randint

param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': [3, 5, 10, 15, None],
    'min_samples_split': randint(2, 20),
    'max_features': ['sqrt', 'log2'],
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42), param_dist,
    n_iter=50, cv=5, scoring='accuracy', n_jobs=-1, random_state=42
)
random_search.fit(X_tr_s, y_tr)

print(f'Best parameters:  {random_search.best_params_}')
print(f'Best CV accuracy: {random_search.best_score_:.4f}')
print(f'Test accuracy:    {random_search.score(X_te_s, y_te):.4f}')

In [ ]:
results_df = pd.DataFrame(grid_search.cv_results_)

pivot = results_df.pivot_table(
    values='mean_test_score',
    index='param_max_depth',
    columns='param_n_estimators',
    aggfunc='max'
)

plt.figure(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd')
plt.title('GridSearch Results — Accuracy by max_depth × n_estimators')
plt.show()

---
## 6 — Learning Curves

Plot train and validation scores as training set size increases.  
Diagnose **underfitting** vs **overfitting** at a glance.

```
Underfitting:  both scores low and close together
Good fit:      both scores high and close together  
Overfitting:   train score high, validation score low (big gap)
```

In [ ]:
def plot_learning_curve(model, X, y, title, ax):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=5, scoring='accuracy',
        train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
    )
    
    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)
    
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='red')
    ax.plot(train_sizes, train_mean, 'o-', color='blue', label='Train')
    ax.plot(train_sizes, val_mean, 'o-', color='red', label='Validation')
    ax.set_xlabel('Training set size')
    ax.set_ylabel('Accuracy')
    ax.set_title(title)
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_learning_curve(
    LogisticRegression(max_iter=5000, C=0.001), X_tr_s, y_tr,
    'Underfitting (high bias)', axes[0]
)

plot_learning_curve(
    RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42), X_tr_s, y_tr,
    'Good Fit', axes[1]
)

plot_learning_curve(
    DecisionTreeClassifier(random_state=42), X_tr_s, y_tr,
    'Overfitting (high variance)', axes[2]
)

plt.suptitle('Learning Curves — Diagnosing Model Problems', fontsize=13)
plt.tight_layout()
plt.show()

---
## 7 — Bias-Variance Tradeoff

The big picture of model performance.

$$\text{Total Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Noise}$$

| Problem | Symptom | Cause | Fix |
|---------|---------|-------|-----|
| **High Bias** (underfitting) | Low train AND test score | Model too simple | More features, complex model, less regularization |
| **High Variance** (overfitting) | High train, low test | Model too complex | More data, regularization, simpler model, dropout |

In [ ]:
complexity = np.linspace(0, 10, 100)
bias_sq = 5 * np.exp(-0.5 * complexity)
variance = 0.1 * np.exp(0.5 * complexity)
noise = np.ones_like(complexity) * 0.5
total_error = bias_sq + variance + noise

plt.figure(figsize=(10, 6))
plt.plot(complexity, bias_sq, 'b-', linewidth=2, label='Bias²')
plt.plot(complexity, variance, 'r-', linewidth=2, label='Variance')
plt.plot(complexity, noise, 'g--', linewidth=1, label='Irreducible noise')
plt.plot(complexity, total_error, 'k-', linewidth=2.5, label='Total error')

optimal = complexity[np.argmin(total_error)]
plt.axvline(x=optimal, color='purple', linestyle=':', linewidth=2, label='Sweet spot')

plt.xlabel('Model Complexity →')
plt.ylabel('Error')
plt.title('Bias-Variance Tradeoff')
plt.legend(fontsize=10)

plt.annotate('UNDERFITTING\n(high bias)', xy=(1, 4), fontsize=11, ha='center', color='blue')
plt.annotate('OVERFITTING\n(high variance)', xy=(8.5, 4), fontsize=11, ha='center', color='red')

plt.grid(True, alpha=0.3)
plt.show()

---
## 8 — Class Imbalance

When 95% of data is one class, a model that always predicts the majority class gets 95% accuracy.  
That model is **useless**.

Strategies:
1. **Class weights**: penalize mistakes on minority class more
2. **Oversampling**: duplicate minority samples (or SMOTE)
3. **Undersampling**: reduce majority samples
4. **Use F1/AUC** instead of accuracy

In [ ]:
X_imb, y_imb = make_classification(
    n_samples=1000, n_features=20, n_informative=5,
    weights=[0.95, 0.05], random_state=42
)

print(f'Class distribution: {np.bincount(y_imb)}')
print(f'Minority class: {np.mean(y_imb == 1):.1%}')

X_itr, X_ite, y_itr, y_ite = train_test_split(X_imb, y_imb, test_size=0.3, random_state=42, stratify=y_imb)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, weights, title in zip(axes,
    [None, 'balanced'],
    ['No class weights — misses minority', 'Balanced weights — catches minority']):
    
    lr = LogisticRegression(class_weight=weights, max_iter=5000)
    lr.fit(X_itr, y_itr)
    y_pred_i = lr.predict(X_ite)
    
    cm = confusion_matrix(y_ite, y_pred_i)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    
    f1 = f1_score(y_ite, y_pred_i)
    recall = recall_score(y_ite, y_pred_i)
    ax.set_title(f'{title}\nF1={f1:.3f}, Recall={recall:.3f}')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.utils import resample

X_train_all = np.column_stack([X_itr, y_itr])
majority = X_train_all[X_train_all[:, -1] == 0]
minority = X_train_all[X_train_all[:, -1] == 1]

minority_upsampled = resample(minority, replace=True, n_samples=len(majority), random_state=42)
upsampled = np.vstack([majority, minority_upsampled])

X_up = upsampled[:, :-1]
y_up = upsampled[:, -1].astype(int)

print(f'Before oversampling: {np.bincount(y_itr)}')
print(f'After oversampling:  {np.bincount(y_up)}')

lr_up = LogisticRegression(max_iter=5000).fit(X_up, y_up)
y_pred_up = lr_up.predict(X_ite)
print(f'\nOversampled model → F1: {f1_score(y_ite, y_pred_up):.3f}, Recall: {recall_score(y_ite, y_pred_up):.3f}')

---
## Evaluation Cheat Sheet

| What you want to know | Use this |
|----------------------|----------|
| Overall accuracy | `accuracy_score` (but beware class imbalance!) |
| Types of errors | Confusion matrix |
| Don't miss positives | Recall |
| Don't make false alarms | Precision |
| Balance precision/recall | F1 score |
| Compare models at all thresholds | ROC curve + AUC |
| Reliable performance estimate | Cross-validation |
| Best hyperparameters | GridSearchCV / RandomizedSearchCV |
| Under/overfitting diagnosis | Learning curves |

**Next**: Feature engineering — the art of making raw data useful →